## Stage 3b: Build patient-level vital features
- Import libraries
- load vitals.filtered.csv and icu_stay_features_stage2.csv
- Build feature set using groupby and reshape data
- Flatten the column names and reset index
- Merge df_stage2 with vitals_features
- Save stage 3 csv into icu_stay_features_stage3.csv

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
notebook_dir = os.getcwd() 
project_root = os.path.dirname(notebook_dir)  
output_dir = os.path.join(project_root, 'data', 'processed') 
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Load vitals_filtered.csv
df_vitals = pd.read_csv(
    os.path.join(output_dir, 'vitals_filtered.csv')
)

In [ ]:
# Load icu_stay_features_stage2.csv
df_stage2 = pd.read_csv(
    os.path.join(output_dir, 'icu_stay_features_stage2.csv')
)

In [ ]:
# Build feature set
vital_features = (
    df_vitals
    .groupby(['stay_id', 'vital_name'])['valuenum']
    .agg(['mean', 'min', 'max', 'std'])
    .round(2)
)

In [ ]:
# Reshape the data
vital_features = (
    vital_features.unstack()
)

In [ ]:
# Flatten the column names
vital_features.columns = [
    f"{vital}_{stat}"
    for stat, vital in vital_features.columns
]

In [ ]:
# Reset the index
vital_features = vital_features.reset_index()

In [ ]:
print(vital_features.head())
print(vital_features.columns)

In [ ]:
print(vital_features.shape)

In [ ]:
vital_features.isnull().sum()

In [ ]:
df_stage3 = df_stage2.merge(
    vital_features,
    on='stay_id',
    how='left'
)

In [ ]:
print(df_stage3.info())

In [ ]:
df_stage3.describe(include="all")

In [ ]:
print(df_stage3.shape)

In [ ]:
print(df_stage3.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# Save the extracted vitals to a CSV file
output_file = os.path.join(output_dir, 'icu_stay_features_stage3.csv')
df_stage3.to_csv(output_file, index=False)

### Summary
Tis notebook transformed cleaned vital sign measurements into patient-level features suitable for machine learning.

Key tasks completed:
- Extracted five clinically relevant vital signs
- Removed physiologically implausible values using predefined clinical ranges
- Aggregated measurements into mean, minimum, maximum, and standard deviation for each ICU stay
- Produced one row per ICU stay
- Merged the aggregated vital features with the Stage 2 patient dataset

The resulting dataset **(icu_stay_features_stage3.csv)** will be used for patient similarity search and downstream machine learning tasks.